# 两支基的去向，决定整张网格

固定矩阵 A=[[1,1],[0,1]]，让输入点在平面上移动。它保留第二个坐标，把第一个坐标改成两个坐标之和。先算 (2,1)、(1,0)、(0,1) 各被送到哪里，再运行。

实验依赖 NumPy、Matplotlib；ipywidgets 提供滑块。没有滑块时可以直接改函数参数。请从第一格开始顺序运行。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon

A = np.array([[1., 1.], [0., 1.]])
e1 = np.array([1., 0.])
e2 = np.array([0., 1.])
x = np.array([2., 1.])
print('A e1 =', A @ e1)
print('A e2 =', A @ e2)
print('A x  =', A @ x)
print('2 A e1 + A e2 =', 2 * (A @ e1) + A @ e2)


## 从两支基，走到整张网格

原点、e1、e1+e2、e2 围成单位正方形。先预测：它们的输出还围成正方形吗？水平网格线和竖直网格线各变成什么？

下面用矩阵 [[1,k],[0,h]]。左图保留输入网格，右图对同一批点逐一执行矩阵乘法。蓝色、橙色分别跟踪第一支、第二支基；绿色区域跟踪单位正方形；红点跟踪当前输入。虚线表示把红点拆成两段基向量组合。两图的坐标刻度相同。

In [ ]:
def inspect_grid(k=1.0, h=1.0, s=2.0, t=1.0):
    matrix = np.array([[1., k], [0., h]])
    point = np.array([s, t])
    square = np.array([[0., 0.], [1., 0.], [1., 1.], [0., 1.]])
    segment = np.linspace(-2, 2, 100)
    fig, axes = plt.subplots(1, 2, figsize=(11.2, 4.6), dpi=160, layout='constrained')
    for ax, mapping, title, labels in [
        (axes[0], np.eye(2), 'Input grid', ('x1', 'x2')),
        (axes[1], matrix, f'Output grid: k={k:g}, h={h:g}', ('b1', 'b2'))
    ]:
        for level in range(-2, 3):
            horizontal = mapping @ np.vstack([segment, np.full_like(segment, level)])
            vertical = mapping @ np.vstack([np.full_like(segment, level), segment])
            ax.plot(*horizontal, color='#0072B2', alpha=0.22, linewidth=1)
            ax.plot(*vertical, color='#D55E00', alpha=0.22, linewidth=1)
        ax.add_patch(Polygon(square @ mapping.T, closed=True, facecolor='#19856F',
                            edgecolor='#19856F', alpha=0.18, linewidth=2))
        for basis, color, name, width in [
            (e1, '#0072B2', 'first basis vector', 0.009),
            (e2, '#D55E00', 'second basis vector', 0.0055)
        ]:
            image = mapping @ basis
            ax.quiver(0, 0, *image, angles='xy', scale_units='xy', scale=1,
                      color=color, width=width, label=name, zorder=4)
        first_leg = mapping @ np.array([[0., s], [0., 0.]])
        second_leg = mapping @ np.array([[s, s], [0., t]])
        ax.plot(*first_leg, '--', color='#0072B2', linewidth=1.5)
        ax.plot(*second_leg, '--', color='#D55E00', linewidth=1.5)
        mapped_point = mapping @ point
        ax.scatter(*mapped_point, s=70, color='#B83F4E', edgecolor='white', zorder=5)
        ax.annotate(f'({mapped_point[0]:g}, {mapped_point[1]:g})', mapped_point,
                    xytext=(-6 if mapped_point[0] > 3.5 else 6,
                            -6 if mapped_point[1] > 2.5 else 6),
                    textcoords='offset points',
                    ha='right' if mapped_point[0] > 3.5 else 'left',
                    va='top' if mapped_point[1] > 2.5 else 'bottom',
                    fontsize=10, color='#B83F4E')
        ax.set(xlim=(-5.5, 5.5), ylim=(-3.5, 3.5),
               xlabel=labels[0], ylabel=labels[1], title=title)
        ax.set_aspect('equal', adjustable='box')
        ax.axhline(0, color='#7F8E98', linewidth=0.6)
        ax.axvline(0, color='#7F8E98', linewidth=0.6)
        ax.legend(loc='lower left', fontsize=8)
    plt.show()
    print('A =\n', matrix)
    print('basis outputs:', matrix @ e1, matrix @ e2)
    print('input:', point, '-> output:', matrix @ point)
    print('by basis combination:', s * (matrix @ e1) + t * (matrix @ e2))

inspect_grid()


## 一次改一个量

1. 固定 h=1，只改变 k。第二支基的输出是 (k,1)，网格怎样跟着它变化？
2. 固定 k=1，把 h 从 1 改成 0。两支基的输出越来越接近，最后会怎样？
3. 固定矩阵，只改输入 s、t。先手算 s*A*e1+t*A*e2，再看红点的输出。

如果没有滑块，修改上面 inspect_grid() 的参数，例如 inspect_grid(k=-1,h=1,s=1,t=2)，重新运行即可。

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display
except ImportError:
    print('滑块需要 ipywidgets；也可直接修改 inspect_grid() 的参数。')
else:
    display(widgets.interactive(
        inspect_grid,
        k=widgets.FloatSlider(value=1, min=-1.5, max=1.5, step=0.25, description='k'),
        h=widgets.FloatSlider(value=1, min=0, max=1.5, step=0.25, description='h'),
        s=widgets.FloatSlider(value=2, min=-2, max=2, step=0.25, description='s'),
        t=widgets.FloatSlider(value=1, min=-2, max=2, step=0.25, description='t')))


## 压到一条线，哪些输入被合到一起？

把 h 改成 0，得到 B=[[1,1],[0,0]]。先预测输入 (1,0) 与 (0,1) 各得到什么，再运行。

试着再找两组不同的输入，让输出相同。用 (s+t,0) 解释：为何同一条 s+t=常数 的直线上的输入，都会被合到同一个输出？图只画有限区域；所有输出恰好组成横轴，需要用这个式子证明。

In [ ]:
B = np.array([[1., 1.], [0., 0.]])
inspect_grid(k=1, h=0)
first = np.array([1., 0.])
second = np.array([0., 1.])
print('first input ->', B @ first)
print('second input ->', B @ second)
print('different inputs, same output?', np.array_equal(B @ first, B @ second))


## 检查相加和数乘

先相加再变换，与先分别变换再相加，应该得到什么？把输入乘 -2 后，输出应怎样变化？先手算，再运行下面的检查。

有限个数值例子只能帮我们观察；对所有输入成立的证明，来自矩阵每一行中的乘法分配律。

In [ ]:
u = np.array([2., 1.])
v = np.array([-1., 2.])
c = -2.
print('A(u+v) =', A @ (u + v))
print('Au+Av  =', A @ u + A @ v)
print('A(cu)  =', A @ (c * u))
print('c(Au)  =', c * (A @ u))


## 多加一个平移，会怎样？

把输出再向右移动 1，即 F(x)=Ax+(1,0)。先预测原点的输出，再预测 F(u+v) 是否仍等于 F(u)+F(v)。每次调用 F 都多加一次平移，这会怎样改变两边？

In [ ]:
shift = np.array([1., 0.])
def shifted(x):
    return A @ x + shift

print('F(0)       =', shifted(np.zeros(2)))
print('F(u+v)     =', shifted(u + v))
print('F(u)+F(v)  =', shifted(u) + shifted(v))


## 从基的去向，重新写出矩阵

另一个线性变换把 e1 送到 (2,1)，把 e2 送到 (1,-1)。先自己写出矩阵，再把 (3,2) 拆成 3e1+2e2，手算输出。

最后离开代码，用自己的话解释：为什么剪切和压扁都满足线性规则，却只有压扁会让不同输入合到一起？

In [ ]:
first_image = np.array([2., 1.])
second_image = np.array([1., -1.])
C = np.column_stack([first_image, second_image])
candidate = np.array([3., 2.])
print('matrix from basis images =\n', C)
print('C @ input =', C @ candidate)
print('by basis images =', 3 * first_image + 2 * second_image)
